# 01 - Extraccion de Datos

**Proyecto:** Pochoclo Predict
**Materia:** Web Mining - Trabajo Practico Final

## Objetivo de este notebook

Construir el dataset crudo de peliculas que se utilizara en todo el proyecto,
consultando la API publica de [The Movie Database (TMDB)](https://www.themoviedb.org/documentation/api).

La variable objetivo (target) del proyecto es **`nota_promedio`**: el promedio
de calificacion de los usuarios de TMDB para cada pelicula, en una escala de
1 a 10. El objetivo final del TP es predecir esta nota a partir de atributos
disponibles *antes o al momento del estreno* (presupuesto, duracion, genero,
elenco, sinopsis, etc.), es decir, un problema de **regresion supervisada**.

## Condiciones del dataset

- Solo peliculas estrenadas **a partir de 1980**.
- Solo peliculas con **director identificado** y con **presupuesto cargado**
  en TMDB (`presupuesto > 0`): son variables centrales para el modelado (NB03
  usa el presupuesto como feature directa, y el director como base de la
  feature historica `director_avg_revenue`).
- Piso minimo de votos (`MIN_VOTE_COUNT`) para que `nota_promedio` sea
  estadisticamente confiable (1 o 2 votos con nota 10 es ruido, no señal).

## Metodologia de muestreo: aleatorio, estratificado por anio, filtrado en linea

Una primera version de este notebook ordenaba `/discover/movie` por
`vote_count.desc` y recorria las paginas de forma secuencial. Eso resultaba
en una muestra **sesgada**: siempre trae las peliculas mas votadas de toda
la historia, que son sistematicamente las mas consagradas/queridas (la nota
promedio del dataset terminaba concentrada alrededor de 7.0). Una segunda
version introdujo aleatoriedad (paginas al azar dentro de cada anio) pero
seguia pidiendo un lote fijo de candidatas al azar y descartando *despues*
las que no tenian presupuesto cargado -el 67% de TMDB no lo tiene-, dejando
el tamaño final del dataset librado al azar de cuantas sobrevivian.

La version actual (`TMDBClient.sample_movies`, en `src/tmdb_client.py`)
corrige esto aplicando el filtro **en linea**, durante la propia busqueda:
para cada anio se piden candidatas al azar (paginas en orden aleatorio,
resultados dentro de cada pagina tambien en orden aleatorio) y se sigue
pidiendo detalle hasta **alcanzar la cantidad objetivo de peliculas que
cumplen** las condiciones (`TARGET_PER_YEAR`) -o hasta agotar el universo
disponible de ese anio, lo que ocurra primero-. Esto maximiza el
aprovechamiento del universo real de peliculas elegibles, en lugar de
sesgar el tamaño final a lo que sobreviva de un lote arbitrario.

La API de TMDB no permite filtrar por "tiene presupuesto cargado" en el
endpoint de descubrimiento (ese dato solo se conoce al pedir el detalle de
la pelicula), por lo que pedir detalle de candidatas que luego se descartan
es inevitable; lo que se evita es frenar la busqueda antes de tiempo.

Por cada pelicula aceptada se pide el detalle completo en **una sola
llamada**, combinando con `append_to_response` la informacion de
reparto/equipo (`credits`), palabras clave (`keywords`) y traducciones
(`translations`) -esta ultima para poder recuperar tambien la sinopsis en
ingles, que se usa mas adelante para el analisis de sentimiento (NB03).


In [4]:
import sys
from pathlib import Path

# Se agrega la raiz del proyecto al sys.path para poder importar el paquete `src`.
# Path.cwd() es la carpeta del propio notebook (Jupyter la fija como cwd por
# defecto), por lo que esto funciona sin importar en que ubicacion del disco
# este copiado el proyecto.
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import os
import pandas as pd

from src import paths
from src.tmdb_client import TMDBClient

pd.set_option("display.max_columns", None)


## 1. Configuracion y conexion a la API

La API Key se lee desde un archivo `.env` en la raiz del proyecto (no se
hardcodea en el codigo, para poder compartir/versionar el proyecto sin
exponer credenciales). Ver `.env.example` para el formato esperado.


In [ ]:
API_KEY = os.environ.get("TMDB_API_KEY")
client = TMDBClient(api_key=API_KEY, language="es-ES")

# Parametros del muestreo
MIN_VOTE_COUNT = 30     # piso de votos para considerar confiable la nota_promedio
YEAR_FROM = 1980
YEAR_TO = 2025
TARGET_PER_YEAR = 100   # peliculas que CUMPLEN el filtro que se buscan por cada anio
SEED = 42               # semilla fija para que el muestreo sea reproducible

print("Cliente TMDB inicializado correctamente.")


## 2. Muestreo y descarga (aleatorio, estratificado por anio, filtrado en linea)

Esta celda tarda bastante (del orden de una hora): por cada pelicula
candidata (cumpla o no el filtro) se pide su detalle completo a la API, y
para los anios con mas peliculas cargadas en TMDB hay que revisar varios
cientos de candidatas para llegar a `TARGET_PER_YEAR` aceptadas.

Para no perder el trabajo ya hecho ante un corte (cierre accidental,
suspension de la maquina, limite de tiempo de un entorno en la nube, etc.),
el progreso se **guarda incrementalmente anio por anio** en un archivo
intermedio (`_peliculas_raw_parcial.csv`). Si este notebook se vuelve a
ejecutar, retoma desde el primer anio que todavia no este guardado en lugar
de arrancar de cero.


In [ ]:
RAW_PARCIAL = paths.DATA_RAW / "_peliculas_raw_parcial.csv"
paths.DATA_RAW.mkdir(parents=True, exist_ok=True)

anios_completados = set()
if RAW_PARCIAL.exists():
    anios_completados = set(pd.read_csv(RAW_PARCIAL, encoding="utf-8-sig")["anio_estreno"].dropna().astype(int).unique())
    print(f"Retomando extraccion: ya hay {len(anios_completados)} anios guardados de una corrida anterior.")
else:
    print("Arrancando extraccion desde cero.")

for anio in range(YEAR_FROM, YEAR_TO + 1):
    if anio in anios_completados:
        continue

    def reportar_avance(a, aceptadas, universo_visto):
        print(f"  {a}: {aceptadas}/{TARGET_PER_YEAR} peliculas aceptadas (de {universo_visto} candidatas revisadas)")

    registros_anio = client.sample_movies(
        year_from=anio,
        year_to=anio,
        min_vote_count=MIN_VOTE_COUNT,
        target_per_year=TARGET_PER_YEAR,
        require_director=True,
        require_budget=True,
        seed=SEED,
        on_year_done=reportar_avance,
    )
    pd.DataFrame(registros_anio).to_csv(
        RAW_PARCIAL, mode="a", header=not RAW_PARCIAL.exists(), index=False, encoding="utf-8-sig"
    )

print("\nExtraccion completa: todos los anios del rango ya estan guardados.")


## 3. Construccion del dataset final

Se arma el DataFrame a partir del archivo incremental y se aplican los
ultimos filtros minimos de calidad (duplicados, `nota_promedio`/
`duracion_min` invalidos). El filtro de director/presupuesto ya se aplico
durante el muestreo (seccion 2).

No se hace aca ninguna transformacion de features (eso corresponde al NB03):
este notebook entrega el dataset **crudo**, tal como llega de la API.


In [ ]:
df = pd.read_csv(RAW_PARCIAL, encoding="utf-8-sig")
df = df.drop_duplicates(subset="id")
df = df[df["nota_promedio"].notna() & (df["nota_promedio"] > 0)]
df = df[df["duracion_min"].fillna(0) > 0]
df = df.reset_index(drop=True)

assert (df["presupuesto"] > 0).all()
assert df["director"].notna().all()

paths.DATA_RAW.mkdir(parents=True, exist_ok=True)
df.to_csv(paths.RAW_MOVIES_CSV, index=False, encoding="utf-8-sig")

print(f"Dataset final: {df.shape[0]} peliculas x {df.shape[1]} columnas")
print(f"Guardado en: {paths.RAW_MOVIES_CSV.relative_to(paths.PROJECT_ROOT)}")


## 4. Verificacion rapida

Ademas de los chequeos habituales, se valida especificamente que el
muestreo haya quedado bien distribuido entre anios (no concentrado en unos
pocos) y que la nota promedio no este artificialmente concentrada en
valores altos.


In [ ]:
df.info()


In [ ]:
df[["nota_promedio", "cantidad_votos", "presupuesto", "recaudacion", "duracion_min", "popularidad"]].describe()


In [ ]:
print(df.groupby("anio_estreno")["id"].count().describe())


In [ ]:
df.head()


## 5. Enriquecimiento: poster de cada pelicula

Para el dashboard final (NB06) conviene mostrar el poster de cada pelicula.
TMDB sirve estas imagenes desde un CDN publico (`image.tmdb.org`): alcanza
con guardar el `poster_path` (un identificador corto, ej. `/qJ2tW6WMU...jpg`)
para poder armar la URL completa al momento de mostrarla -no hace falta
descargar ni guardar los archivos de imagen en el proyecto-.

Esta seccion corre **sobre el dataset ya armado** (no repite el muestreo de
la seccion 2): por cada `id` ya aceptado pide unicamente el poster/backdrop
(`TMDBClient.get_poster_paths`, una llamada liviana sin `append_to_response`)
y lo agrega como columnas nuevas. Al igual que la extraccion principal, el
progreso se guarda incrementalmente (`_posters_parcial.csv`, en lotes de 50)
para poder retomar sin perder lo ya conseguido. Al estar el proyecto dentro
de una carpeta sincronizada con OneDrive, escribir el archivo en cada
iteracion (como hace la extraccion principal, anio por anio) choca a veces
con el bloqueo temporal que OneDrive toma mientras sincroniza: por eso aca
se agrupan las escrituras en lotes y se reintenta ante un bloqueo transitorio.


In [ ]:
import time as _time

POSTERS_PARCIAL = paths.DATA_RAW / "_posters_parcial.csv"
TAMANIO_LOTE = 50


def guardar_lote_con_reintento_en(ruta, lote, intentos=5):
    for intento in range(intentos):
        try:
            pd.DataFrame(lote).to_csv(
                ruta, mode="a", header=not ruta.exists(), index=False, encoding="utf-8-sig"
            )
            return
        except PermissionError:
            if intento == intentos - 1:
                raise
            _time.sleep(2 * (intento + 1))  # OneDrive puede tener el archivo bloqueado un instante


ids_con_poster = set()
if POSTERS_PARCIAL.exists():
    ids_con_poster = set(pd.read_csv(POSTERS_PARCIAL, encoding="utf-8-sig")["id"].unique())
    print(f"Retomando: ya hay posters guardados para {len(ids_con_poster)} peliculas.")

pendientes = [i for i in df["id"] if i not in ids_con_poster]
print(f"Peliculas pendientes de poster: {len(pendientes)} de {len(df)}")

lote_actual = []
for movie_id in pendientes:
    info = client.get_poster_paths(movie_id)
    lote_actual.append({"id": movie_id, **info})
    if len(lote_actual) >= TAMANIO_LOTE:
        guardar_lote_con_reintento_en(POSTERS_PARCIAL, lote_actual)
        lote_actual = []

if lote_actual:
    guardar_lote_con_reintento_en(POSTERS_PARCIAL, lote_actual)

print("Enriquecimiento de posters completo.")


In [ ]:
posters = pd.read_csv(POSTERS_PARCIAL, encoding="utf-8-sig").drop_duplicates(subset="id")

columnas_previas = [c for c in ["poster_path", "backdrop_path"] if c in df.columns]
df = df.drop(columns=columnas_previas).merge(posters, on="id", how="left")
df.to_csv(paths.RAW_MOVIES_CSV, index=False, encoding="utf-8-sig")

print(f"Peliculas con poster disponible: {df['poster_path'].notna().sum()} de {len(df)}")
print(f"Ejemplo de URL: {TMDBClient.poster_url(df['poster_path'].dropna().iloc[0])}")


## 6. Enriquecimiento: identificadores externos (para cruces futuros)

Este proyecto solo usa datos de TMDB, pero deja preparado el cruce con otras
fuentes para una posible iteracion futura (ej. premios, critica especializada):
TMDB expone, sin costo adicional, el `imdb_id` de cada pelicula y su
`wikidata_id` (el QID de Wikidata) a traves del endpoint liviano
`/movie/{id}/external_ids`. Guardando estos dos identificadores ahora, un
futuro enriquecimiento (datasets oficiales de IMDb, consultas SPARQL a
Wikidata por la propiedad `P166` "award received", etc.) podria cruzar por
id exacto en lugar de intentar matchear peliculas por titulo -el enfoque
fragil que usaba el prototipo original de scraping en `legacy/`-.

Mismo patron que la seccion 5: corre sobre el dataset ya armado, con
checkpoint incremental en lotes para poder retomar sin perder progreso.


In [ ]:
EXTERNAL_IDS_PARCIAL = paths.DATA_RAW / "_external_ids_parcial.csv"

ids_con_external = set()
if EXTERNAL_IDS_PARCIAL.exists():
    ids_con_external = set(pd.read_csv(EXTERNAL_IDS_PARCIAL, encoding="utf-8-sig")["id"].unique())
    print(f"Retomando: ya hay identificadores externos guardados para {len(ids_con_external)} peliculas.")

pendientes_external = [i for i in df["id"] if i not in ids_con_external]
print(f"Peliculas pendientes de identificadores externos: {len(pendientes_external)} de {len(df)}")

lote_actual = []
for movie_id in pendientes_external:
    info = client.get_external_ids(movie_id)
    lote_actual.append({"id": movie_id, **info})
    if len(lote_actual) >= TAMANIO_LOTE:
        guardar_lote_con_reintento_en(EXTERNAL_IDS_PARCIAL, lote_actual)
        lote_actual = []

if lote_actual:
    guardar_lote_con_reintento_en(EXTERNAL_IDS_PARCIAL, lote_actual)

print("Enriquecimiento de identificadores externos completo.")


In [ ]:
external_ids = pd.read_csv(EXTERNAL_IDS_PARCIAL, encoding="utf-8-sig").drop_duplicates(subset="id")

columnas_previas = [c for c in ["imdb_id", "wikidata_id"] if c in df.columns]
df = df.drop(columns=columnas_previas).merge(external_ids, on="id", how="left")
df.to_csv(paths.RAW_MOVIES_CSV, index=False, encoding="utf-8-sig")

print(f"Peliculas con imdb_id: {df['imdb_id'].notna().sum()} de {len(df)}")
print(f"Peliculas con wikidata_id: {df['wikidata_id'].notna().sum()} de {len(df)}")


## 7. Enriquecimiento: filmografia completa de director y reparto

Las features historicas del NB03 (desempeño previo del director y del
elenco) se calculaban solo con las peliculas de **nuestra propia muestra**:
como la muestra es una fraccion del catalogo, el 68% de los directores no
tenia ninguna pelicula previa dentro de ella y la feature quedaba vacia.

Para resolverlo se pide a TMDB la **filmografia completa** de cada director
y de cada uno de los 3 actores principales (`/person/{id}/movie_credits`),
incluyendo peliculas que no estan en la muestra. Son dos pasos:

1. Por cada pelicula, los ids de su director y de sus 3 actores principales
   (`/movie/{id}/credits`), con el mismo criterio que `parse_movie`.
2. Por cada persona distinta, su filmografia: fecha de estreno, nota y
   cantidad de votos de cada pelicula que dirigio o en la que actuo.

El calculo de las features (solo con peliculas *anteriores* a cada estreno)
se hace en el NB03. Aca solo se descargan los datos.

Como son varios miles de consultas, se hacen de a 6 en paralelo (bastante
por debajo del limite de la API) y se guardan por lotes con checkpoint,
igual que en las secciones anteriores.


In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor


def enriquecer_en_paralelo(ids, funcion, ruta_checkpoint, columna_id, workers=6, tamanio_lote=60):
    hechos = set()
    if ruta_checkpoint.exists():
        hechos = set(pd.read_csv(ruta_checkpoint, encoding="utf-8-sig")[columna_id].unique())
    pendientes = [i for i in ids if i not in hechos]
    print(f"Pendientes: {len(pendientes)} de {len(ids)} (ya guardados: {len(hechos)})")

    with ThreadPoolExecutor(max_workers=workers) as pool:
        for inicio in range(0, len(pendientes), tamanio_lote):
            lote = pendientes[inicio:inicio + tamanio_lote]
            guardar_lote_con_reintento_en(ruta_checkpoint, list(pool.map(funcion, lote)))

    return pd.read_csv(ruta_checkpoint, encoding="utf-8-sig").drop_duplicates(subset=columna_id)


CREDITOS_PARCIAL = paths.DATA_RAW / "_creditos_ids_parcial.csv"
creditos = enriquecer_en_paralelo(
    df["id"].tolist(),
    lambda movie_id: {"id": movie_id, **client.get_credit_ids(movie_id)},
    CREDITOS_PARCIAL,
    columna_id="id",
)

columnas_previas = [c for c in ["director_id", "reparto_ids"] if c in df.columns]
df = df.drop(columns=columnas_previas).merge(creditos, on="id", how="left")
df["director_id"] = df["director_id"].astype("Int64")
df.to_csv(paths.RAW_MOVIES_CSV, index=False, encoding="utf-8-sig")
print(f"Peliculas con director_id: {df['director_id'].notna().sum()} de {len(df)}")


In [ ]:
personas = set(df["director_id"].dropna().astype(int))
for lista in df["reparto_ids"].dropna():
    personas.update(json.loads(lista))
print(f"Personas distintas (directores + actores principales): {len(personas)}")

FILMOGRAFIAS_PARCIAL = paths.DATA_RAW / "_filmografias_parcial.csv"
filmografias = enriquecer_en_paralelo(
    sorted(personas),
    client.get_person_filmography,
    FILMOGRAFIAS_PARCIAL,
    columna_id="person_id",
)
filmografias.to_csv(paths.FILMOGRAFIAS_CSV, index=False, encoding="utf-8-sig")

peliculas_por_persona = filmografias["como_director"].apply(lambda s: len(json.loads(s))) + \
                        filmografias["como_actor"].apply(lambda s: len(json.loads(s)))
print(f"Filmografias guardadas: {len(filmografias)} personas, {peliculas_por_persona.sum():,} peliculas en total")
print(f"Guardado en: {paths.FILMOGRAFIAS_CSV.relative_to(paths.PROJECT_ROOT)}")
